# CEG-WM Content V9 Stability — formal user-only GPU handoff

This Colab notebook executes the frozen initial invocation only for `content-v9-stability-9bc8a94c1d02-63c17e8200a9-805bc21e173a` from `stage-a-content-v9-multi-cohort-stability-evaluation@26151aad22402c8d1df41dacf2bb140391e8e349`. It mounts Drive first, proves the fresh exact checkout, installs only that checkout, validates the public protocol and accepted calibration-asset identities, reads both Secrets, and invokes the existing formal runner exactly once.

The frozen old-roster, current-V6-roster, and novel-seed sections remain independent inside the runner-owned artifact. This notebook does not implement their Gates, pool sections, or interpret scientific outcomes.

Create Colab Secrets named `CEG_WM_ROOT_KEY` and `HF_TOKEN`. Run the cells once from top to bottom. Stop after any failure or interruption.


## 1. Mount Drive, then prove the fresh execution checkout

Drive mounting is the first external action. The source path and exact-bound local and Drive run destinations must all be absent before checkout or installation.

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
except BaseException:
    print('CEGWM_CONTENT_V9_STABILITY_HANDOFF_FAILURE {"error_class":"OtherOperationalError","execution_exact":"26151aad22402c8d1df41dacf2bb140391e8e349","run_id":"content-v9-stability-9bc8a94c1d02-63c17e8200a9-805bc21e173a","stage":"drive_mount","status":"operational_failure"}', flush=True)
    HANDOFF_FAILED = True
else:
    HANDOFF_FAILED = False

import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-v9-multi-cohort-stability-evaluation"
EXACT = "26151aad22402c8d1df41dacf2bb140391e8e349"
RUNNER_MODULE = "experiments.run_content_v9_stability"
FAILURE_PREFIX = "CEGWM_CONTENT_V9_STABILITY_HANDOFF_FAILURE"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V9_STABILITY_ARTIFACT"
METHOD_ID = "content_v9_v6_calibrated_weighted_joint_multi_cohort_stability_v1"
CANDIDATE_ID = "content_v9_v6_calibrated_weighted_joint_multi_cohort_stability_semantic_gate_v1"
PROTOCOL_ID = "cegwm-stage-a-content-v9-calibrated-weighted-joint-multi-cohort-stability-v1"
PROTOCOL_DIGEST = "9bc8a94c1d022cfaaf3c36018422b245e42764571314ee048d612e58a19ca031"
RUN_ID = "content-v9-stability-9bc8a94c1d02-63c17e8200a9-805bc21e173a"
PUBLIC_KEY_DIGEST = "805bc21e173a83898f3b7034d75e6ed02f65894a6885377d9659ee3091b4dd77"
ROSTER_SHA256 = "d9dd998fb3e7e3e0c4693d1188fd18cc7df32424f47ee378b385cabcee51ceb2"
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
DINO_ASSET_ID = "facebook/dinov2-small"
CALIBRATION_ASSET_SHA256 = "63c17e8200a92383b061541fc234dfef36e4b7356954c160ce5f048f820cde96"
CALIBRATION_SIDECAR_FILE_SHA256 = "d543d604e5d9226ddb4c378e160fa389abce223fe8adbb54562c3e6666537301"

repo = pathlib.Path("/content/cegwm-stage-a-content-v9-stability-source")
local_work_root = pathlib.Path("/content/cegwm-stage-a-content-v9-stability-local")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v9_multi_cohort_stability_evaluation")
bound_local_run = local_work_root / RUN_ID
bound_drive_run = artifact_sink / RUN_ID
RUNNER_ATTEMPTED = False

_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "UnicodeDecodeError", "ValueError",
}

def fail(stage, error_class="RuntimeError"):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    HANDOFF_FAILED = True
    payload = {
        "status": "operational_failure",
        "run_id": RUN_ID,
        "execution_exact": EXACT,
        "stage": stage,
        "error_class": error_class,
    }
    line = FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":"))
    if len(line.encode("utf-8")) <= 4096:
        print(line, flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists() or bound_local_run.exists() or bound_drive_run.exists():
            raise FileExistsError
        subprocess.run(
            ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError
    except BaseException as error:
        fail("source_checkout_identity_validation", type(error).__name__)

## 2. Install only the checked-out project

This installs only the project declared by the frozen checkout, then rechecks the named branch, exact revision, clean state, and initial-only destinations.

In [ ]:
if not HANDOFF_FAILED:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError
    except BaseException as error:
        fail("dependency_install_and_source_recheck", type(error).__name__)

## 3. Validate public identities and invoke the stability runner once

The root key and Hugging Face token enter only the child environment and are cleared promptly. Raw child stdout and stderr are never printed. The stability runner remains the sole writer of checkpoint and terminal artifact pairs.


In [ ]:
import hashlib
import os

if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    normalized_key = b""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    CAPTURE_LIMIT = 8192
    try:
        from google.colab import userdata
        from cegwm.protocol.content_chain_v9_stability import (
            CONTENT_V9_STABILITY_CALIBRATION_ASSET,
            CONTENT_V9_STABILITY_CALIBRATION_ASSET_SHA256,
            CONTENT_V9_STABILITY_CALIBRATION_ASSET_SIDECAR_FILE_SHA256,
            CONTENT_V9_STABILITY_CURRENT_MANIFEST,
            CONTENT_V9_STABILITY_CURRENT_MANIFEST_SHA256,
            CONTENT_V9_STABILITY_EVALUATED_CANDIDATE_ID,
            CONTENT_V9_STABILITY_METHOD_ID,
            CONTENT_V9_STABILITY_NOVEL_MANIFEST,
            CONTENT_V9_STABILITY_NOVEL_MANIFEST_SHA256,
            CONTENT_V9_STABILITY_OLD_MANIFEST,
            CONTENT_V9_STABILITY_OLD_MANIFEST_SHA256,
            CONTENT_V9_STABILITY_PROTOCOL_DIGEST,
            CONTENT_V9_STABILITY_PROTOCOL_ID,
            deterministic_stability_run_id,
            load_content_v9_stability_contract,
        )
        from cegwm.shared.keys import normalize_detection_key, public_key_digest

        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
            or bound_local_run.exists()
            or bound_drive_run.exists()
        ):
            raise RuntimeError

        config_root = repo / "configs" / "content_chain"
        contract = load_content_v9_stability_contract(repo)
        asset_path = config_root / CONTENT_V9_STABILITY_CALIBRATION_ASSET
        asset_sidecar_path = asset_path.with_name(asset_path.name + ".sha256")
        identities = contract.config["identities"]
        generation_runtime = contract.v6_protocol.config["generation_runtime"]
        content_analysis = contract.v6_protocol.config["content_analysis"]
        if (
            contract.config["protocol_id"] != PROTOCOL_ID
            or contract.protocol_digest != PROTOCOL_DIGEST
            or CONTENT_V9_STABILITY_PROTOCOL_ID != PROTOCOL_ID
            or CONTENT_V9_STABILITY_PROTOCOL_DIGEST != PROTOCOL_DIGEST
            or identities["method_id"] != METHOD_ID
            or CONTENT_V9_STABILITY_METHOD_ID != METHOD_ID
            or identities["evaluated_candidate_id"] != CANDIDATE_ID
            or CONTENT_V9_STABILITY_EVALUATED_CANDIDATE_ID != CANDIDATE_ID
            or len(contract.old_roster_reference) != 8
            or len(contract.current_v6_roster_reference) != 8
            or len(contract.novel_seed_01) != 32
            or len(contract.novel_seed_02) != 32
            or hashlib.sha256((config_root / CONTENT_V9_STABILITY_OLD_MANIFEST).read_bytes()).hexdigest() != CONTENT_V9_STABILITY_OLD_MANIFEST_SHA256
            or hashlib.sha256((config_root / CONTENT_V9_STABILITY_CURRENT_MANIFEST).read_bytes()).hexdigest() != CONTENT_V9_STABILITY_CURRENT_MANIFEST_SHA256
            or hashlib.sha256((config_root / CONTENT_V9_STABILITY_NOVEL_MANIFEST).read_bytes()).hexdigest() != CONTENT_V9_STABILITY_NOVEL_MANIFEST_SHA256
            or ROSTER_SHA256 != CONTENT_V9_STABILITY_NOVEL_MANIFEST_SHA256
            or generation_runtime["model_id"] != MODEL_ID
            or content_analysis["asset_id"] != DINO_ASSET_ID
            or CONTENT_V9_STABILITY_CALIBRATION_ASSET_SHA256 != CALIBRATION_ASSET_SHA256
            or CONTENT_V9_STABILITY_CALIBRATION_ASSET_SIDECAR_FILE_SHA256 != CALIBRATION_SIDECAR_FILE_SHA256
            or hashlib.sha256(asset_path.read_bytes()).hexdigest() != CALIBRATION_ASSET_SHA256
            or hashlib.sha256(asset_sidecar_path.read_bytes()).hexdigest() != CALIBRATION_SIDECAR_FILE_SHA256
        ):
            raise RuntimeError

        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip():
            raise RuntimeError
        if not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        normalized_key = normalize_detection_key(root_key)
        key_digest = public_key_digest(normalized_key)
        normalized_key = b""
        if (
            key_digest != PUBLIC_KEY_DIGEST
            or RUN_ID != deterministic_stability_run_id(
                contract.protocol_digest, CALIBRATION_ASSET_SHA256, key_digest
            )
        ):
            raise RuntimeError

        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {
            name: value for name, value in os.environ.items()
            if not any(marker in name.upper() for marker in secret_markers)
        }
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        process = subprocess.Popen(
            [
                sys.executable, "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
                "--local-work-root", str(local_work_root),
                "--artifact-sink", str(artifact_sink),
            ],
            cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
        )
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, CAPTURE_LIMIT - len(captured))
            captured.extend(chunk[:remaining])
            if len(chunk) > remaining:
                capture_overflow = True
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        normalized_key = b""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None

    captured.clear()
    if launch_error is not None:
        fail("formal_runner_launch", launch_error)
    elif runner_rc != 0:
        fail("formal_runner_nonzero")
    elif capture_overflow:
        fail("formal_runner_stdout_overflow")

## 4. Return existing Drive artifact pairs

This cell is runner-free and read-only. It discovers the existing terminal pair, or every complete contiguous checkpoint pair when no terminal pair exists, and validates each sidecar's exact ZIP filename binding before printing one bounded path-and-declared-hash receipt. It does not read or hash ZIP bytes.


In [ ]:
import json
import pathlib
import re

RUN_ID = "content-v9-stability-9bc8a94c1d02-63c17e8200a9-805bc21e173a"
EXACT = "26151aad22402c8d1df41dacf2bb140391e8e349"
PROTOCOL_DIGEST = "9bc8a94c1d022cfaaf3c36018422b245e42764571314ee048d612e58a19ca031"
CALIBRATION_ASSET_SHA256 = "63c17e8200a92383b061541fc234dfef36e4b7356954c160ce5f048f820cde96"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V9_STABILITY_ARTIFACT"
ARTIFACT_FAILURE_PREFIX = "CEGWM_CONTENT_V9_STABILITY_HANDOFF_FAILURE"
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v9_multi_cohort_stability_evaluation")
run_dir = artifact_sink / RUN_ID
prior_failure = bool(globals().get("HANDOFF_FAILED", False))
artifact_failed = False

def artifact_fail(stage):
    global artifact_failed
    if artifact_failed:
        return
    artifact_failed = True
    if not prior_failure:
        payload = {
            "status": "operational_failure",
            "run_id": RUN_ID,
            "stage": stage,
            "error_class": "RuntimeError",
        }
        line = ARTIFACT_FAILURE_PREFIX + " " + json.dumps(payload, sort_keys=True, separators=(",", ":"))
        if len(line.encode("utf-8")) <= 4096:
            print(line, flush=True)

def validate_pair(archive_path, sidecar_path):
    if not archive_path.is_file() or not sidecar_path.is_file():
        raise RuntimeError
    sidecar_text = sidecar_path.read_text(encoding="ascii")
    match = re.fullmatch(r"([0-9a-f]{64})  ([^\s]+)\n", sidecar_text)
    if match is None or match.group(2) != archive_path.name:
        raise RuntimeError
    return {
        "archive_path": str(archive_path),
        "sidecar_path": str(sidecar_path),
        "sha256": match.group(1),
    }

if not prior_failure and not pathlib.Path("/content/drive/MyDrive").exists():
    artifact_fail("drive_not_mounted")

if not prior_failure and not artifact_failed:
    try:
        terminal_archive = run_dir / (RUN_ID + ".zip")
        terminal_sidecar = run_dir / (RUN_ID + ".zip.sha256")
        terminal_presence = (terminal_archive.exists(), terminal_sidecar.exists())
        if terminal_presence == (True, True):
            artifact_kind = "terminal"
            pairs = [validate_pair(terminal_archive, terminal_sidecar)]
        elif terminal_presence != (False, False):
            raise RuntimeError
        else:
            checkpoint_archives = sorted(run_dir.glob(RUN_ID + ".checkpoint-*.zip"))
            checkpoint_sidecars = sorted(run_dir.glob(RUN_ID + ".checkpoint-*.zip.sha256"))
            if not checkpoint_archives:
                raise RuntimeError
            sequence_to_archive = {}
            for archive_path in checkpoint_archives:
                match = re.fullmatch(re.escape(RUN_ID) + r"\.checkpoint-([0-9]{4})\.zip", archive_path.name)
                if match is None:
                    raise RuntimeError
                sequence_to_archive[int(match.group(1))] = archive_path
            if sorted(sequence_to_archive) != list(range(len(sequence_to_archive))):
                raise RuntimeError
            expected_sidecars = {
                pathlib.Path(str(archive_path) + ".sha256")
                for archive_path in sequence_to_archive.values()
            }
            if set(checkpoint_sidecars) != expected_sidecars:
                raise RuntimeError
            artifact_kind = "checkpoint"
            pairs = [
                validate_pair(sequence_to_archive[index], pathlib.Path(str(sequence_to_archive[index]) + ".sha256"))
                for index in range(len(sequence_to_archive))
            ]
        receipt = {
            "status": "drive_artifacts_ready",
            "run_id": RUN_ID,
            "execution_exact": EXACT,
            "protocol_digest": PROTOCOL_DIGEST,
            "calibration_asset_sha256": CALIBRATION_ASSET_SHA256,
            "artifact_kind": artifact_kind,
            "pairs": pairs,
        }
        receipt_line = ARTIFACT_PREFIX + " " + json.dumps(receipt, sort_keys=True, separators=(",", ":"))
        if len(receipt_line.encode("utf-8")) > 4096:
            raise RuntimeError
        print(receipt_line, flush=True)
    except BaseException:
        artifact_fail("artifact_pair_validation")

## Stop boundary

Return only the bounded Drive artifact receipt and the referenced existing ZIP/SHA-256 pairs, or the single sanitized failure line when no valid pair is available. The notebook does not relaunch the runner, mutate or read archive bytes, expose child streams, reveal Secrets or private state, duplicate section Gates, or interpret scientific outcomes.
